### Libraries

In [ ]:
# pip install pandas
import pandas as pd
# pip install scikit-learn
from sklearn.model_selection import GroupShuffleSplit
# https://scikit-learn.org/0.20/modules/generated/sklearn.preprocessing.OrdinalEncoder.html
from sklearn.preprocessing import OrdinalEncoder
# https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.CategoricalNB.html
from sklearn.naive_bayes import CategoricalNB

### Load Dataset

In [10]:
df = pd.read_parquet("data/lax_flights.parquet")

### Turn Dataset into Bayes Classifier Dataset

In [11]:
# https://pandas.pydata.org/docs/user_guide/timeseries.html
# turn each formal date into a day of the week (such as sun, tue, mon, etc.)
flight_date = pd.to_datetime(df.flightDate)
df["day"] = flight_date.dt.day_name().str[:3]

# calculate the booking window the buyer has. if the buy has a flight date of April 16, and searched for the ticket on April 14, then we do 16-14 = 2
# dt.days turns the number into a plain number (such as 2)
days_ahead = (flight_date - pd.to_datetime(df.searchDate)).dt.days

# https://pandas.pydata.org/docs/reference/api/pandas.cut.html
# bin each flight window the user has. this is for easier data reading.
df["window"] = pd.cut(days_ahead, bins=[0, 7, 21, 45, 60], labels=["1-7", "8-21", "22-45", "46-60"])

# function! sections off all hours into buckets so that we can see each hour as morning, afternoon, or night
def time_bucket(hour):
    if 3 <= hour <= 11: return "Morning"
    if 12 <= hour <= 17: return "Afternoon"
    return "Evening"

# departure string breakdown
# full departure time string
text = df.segmentsDepartureTimeRaw
# we want the hour only
hour_text = text.str[11:13]
# turn into an int
hour = hour_text.astype(int)
# use function above to morning, night, or afternoon
# call function like this bc we need to pass 2 MILLION pieces of data to it
df["time"] = hour.map(time_bucket)

# if there is a stop, 1+, if not, nonstop
df["stops"] = df.isNonStop.map({True: "Nonstop", False: "1+"})

# only keep first 2 chars of the airline (AB || CD turns into AB)
df["airline"] = df.segmentsAirlineCode.str[:2]

# display first 5 rows
df[["day", "time", "window", "stops", "airline"]].head()



,day,time,window,stops,airline
0,Sun,Evening,1-7,1+,F9
1,Sun,Evening,1-7,Nonstop,NK
2,Sun,Afternoon,1-7,Nonstop,NK
3,Sun,Morning,1-7,1+,B6
4,Sun,Afternoon,1-7,1+,UA


### Good Deal vs Bad Deal Training

In [ ]:
# 20% of flights go into testing, random_state = 42 makes sure the state is the same random state each time (this is just for consistency, it's kinda like a seed), n_splits = 1 is how many diff train/test splits we make (1)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=3)

# split our dataset, making sure all rows with the same legID as one group that has to stay tgther
train_idx, test_idx = next(splitter.split(df, groups=df.legId))

# make the train and test tables using the rows we just calculated
# .copy makes it so that we have separate tables instead of just views into df
train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

# shared flights should be 0 if data was split correctly
print(len(train), len(test), "shared flights:", len(set(train.legId) & set(test.legId)))

1606182 406255 shared flights: 0


In [ ]:
# cheapest-third cutoff for each destination, from TRAINING data only
# groups by airport type (such as BOS or ATL), and only looks at the fares
cutoff = train.groupby("destinationAirport").totalFare.quantile(0.33)

# good deal = fare at or below its route's cutoff
train["good_deal"] = train.totalFare <= train.destinationAirport.map(cutoff)
test["good_deal"] = test.totalFare <= test.destinationAirport.map(cutoff)

print(cutoff)
# check how many flights in the training dataset are good deals
print("amt/percentage of good deals:", train.good_deal.mean())

destinationAirport
ATL    328.6
BOS    327.1
LGA    327.2
Name: totalFare, dtype: float64
amt/percentage of good deals: 0.3305254323607163


### Printing out Bayes Classifier Table

In [ ]:
# without random state, it would all be sunday if we printed just .head()
train[["day", "time", "window", "stops", "airline", "good_deal"]].sample(14, random_state=1)

,day,time,window,stops,airline,good_deal
1287237,Mon,Morning,8-21,1+,DL,False
77725,Thu,Morning,8-21,1+,UA,False
1361447,Sat,Afternoon,8-21,1+,AA,True
13301,Mon,Evening,46-60,1+,AA,False
1435547,Sat,Morning,22-45,1+,AA,True
1545898,Fri,Evening,1-7,1+,AA,True
1795917,Sun,Morning,8-21,1+,AA,False
193866,Thu,Morning,46-60,1+,UA,False
1409850,Thu,Evening,22-45,1+,B6,True
620630,Thu,Afternoon,22-45,1+,AA,False
